# County Population Choropleth Workflow

This notebook demonstrates a multi-agent GAS workflow using `GasClient`.

Question: create a county-level choropleth map showing the 2021 population for the contiguous United States. Use 5 quantile classes and Lambert Conformal Conic.

In [ ]:
import sys
from pathlib import Path
from urllib.parse import urljoin

from IPython.display import Image, display

project_root = Path.cwd()
if project_root.name == "examples":
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from gas_client import GasClient


## User Settings

In [ ]:
server_url = "http://127.0.0.1:4042"

openai_api_key = "YOUR_OPENAI_API_KEY"
us_census_demography_key = "918876e93cf30566c02b367fcad29644d861817e"

data_retrieval_agent_id = "geospatial_data_retrieval_agent"
projection_agent_id = "map_projection_agent"
mapping_agent_id = "mapping_agent"

poll_interval = 5
timeout_seconds = 2400

## Helper Functions

In [ ]:

def first_artifact_url(task_result, preferred_formats=None):
    artifacts = task_result.get("outputs", {}).get("artifacts", [])
    if not artifacts:
        raise RuntimeError("The task returned no artifacts.")

    preferred_formats = preferred_formats or []
    for preferred_format in preferred_formats:
        for artifact in artifacts:
            artifact_format = artifact.get("format") or artifact.get("type")
            if artifact.get("url") and str(artifact_format).lower() == preferred_format.lower():
                return artifact["url"]

    for artifact in artifacts:
        if artifact.get("url"):
            return artifact["url"]

    raise RuntimeError("The task returned artifacts, but none had a URL.")


def absolute_url(url):
    if url.startswith("/"):
        return urljoin(server_url, url)
    return url


def display_png_artifacts(task_result):
    artifacts = task_result.get("outputs", {}).get("artifacts", [])
    for artifact in artifacts:
        artifact_url = artifact.get("url")
        filename = artifact.get("filename") or artifact.get("name") or ""
        if artifact_url and str(filename).lower().endswith(".png"):
            display(Image(url=absolute_url(artifact_url)))


## Create the Client and Agents

In [ ]:
client = GasClient(
    server_url,
    openai_api_key=openai_api_key,
)

data_agent = client.agent(data_retrieval_agent_id)
projection_agent = client.agent(projection_agent_id)
mapping_agent = client.agent(mapping_agent_id)

## Step 1: Retrieve County Population Data

In [ ]:

retrieval_task = None
for event in data_agent.execute_task(
    (
        "Download a county-level dataset for the contiguous United States "
        "with county geometries and 2021 total population. Use the supported "
        "US Census Bureau demography source and the Census API key provided "
        "in source_credentials. Exclude Alaska, Hawaii, Puerto Rico, and "
        "other territories. Return one clean geospatial dataset with a "
        "population attribute for 2021 and county identifiers."
    ),
    mode="stream",
    artifact_delivery="URL",
    credentials={
        "source_credentials": {
            "US_Census_demography": {
                "key": us_census_demography_key,
            }
        },
    },
    timeout=timeout_seconds,
):
    client.print_stream_event(event)
    if event.get("event") == "task_result":
        retrieval_task = event.get("payload")

if retrieval_task is None:
    raise RuntimeError("The stream ended before returning a task_result event.")

client.print_task_summary(retrieval_task)


In [ ]:
county_population_url = first_artifact_url(
    retrieval_task,
    preferred_formats=["geojson", "gpkg"],
)

county_population_url

## Step 2: Project to Lambert Conformal Conic

In [ ]:

projection_task = None
for event in projection_agent.execute_task(
    (
        "Project this county population dataset to a Lambert Conformal "
        "Conic coordinate reference system suitable for the contiguous "
        "United States. Use ESRI:102004 as the target CRS. Preserve all "
        "population and county attributes. Return the projected dataset "
        "as GeoJSON or GeoPackage."
    ),
    mode="stream",
    input_datasets=[county_population_url],
    artifact_delivery="URL",
    timeout=timeout_seconds,
):
    client.print_stream_event(event)
    if event.get("event") == "task_result":
        projection_task = event.get("payload")

if projection_task is None:
    raise RuntimeError("The stream ended before returning a task_result event.")

client.print_task_summary(projection_task)


In [ ]:
projected_counties_url = first_artifact_url(
    projection_task,
    preferred_formats=["geojson", "gpkg"],
)

projected_counties_url

## Step 3: Create the Choropleth Map

In [ ]:

map_task = None
for event in mapping_agent.execute_task(
    (
        "Create a county-level choropleth map for the contiguous United "
        "States showing 2021 population. Use the population field for color, "
        "use a quantile classification scheme with 5 classes, and title the "
        "map 'County Population, 2021'. The input dataset is already projected "
        "to Lambert Conformal Conic, so preserve that projected appearance."
    ),
    mode="stream",
    input_datasets=[projected_counties_url],
    artifact_delivery="URL",
    timeout=timeout_seconds,
):
    client.print_stream_event(event)
    if event.get("event") == "task_result":
        map_task = event.get("payload")

if map_task is None:
    raise RuntimeError("The stream ended before returning a task_result event.")

client.print_task_summary(map_task)


## Display the Final Map

In [ ]:
display_png_artifacts(map_task)